# Fine-Tuning IndoBERT untuk Deteksi Ulasan Palsu (Tokopedia)

Notebook ini dibuat untuk memudahkan Anda melakukan *fine-tuning* model Deep Learning tercanggih untuk Bahasa Indonesia: **IndoBERT** (`indobenchmark/indobert-base-p1`) menggunakan **Google Colab GPU Gratis**.

### ⚠️ PENTING: Aktifkan GPU terlebih dahulu!
Sebelum menjalankan sel apa pun, pastikan Anda menggunakan runtime GPU:
1. Di menu atas, pilih **Runtime** -> **Change runtime type** (Ubah tipe runtime).
2. Di bagian **Hardware accelerator** (Akselerator perangkat keras), pilih **T4 GPU** (atau GPU yang tersedia).
3. Klik **Save**.

## 1. Install Library yang Dibutuhkan
Kita akan menginstal pustaka PyTorch, Hugging Face `transformers`, `accelerate` untuk optimalisasi GPU, dan utilitas pendukung lainnya.

In [ ]:
!pip install torch transformers[torch] accelerate -U
!pip install scikit-learn pandas tqdm

## 2. Unggah Dataset Anda
Jalankan sel di bawah ini untuk mengunggah file `tokopedia_preprocessed.csv` dari komputer lokal Anda ke Colab.

In [ ]:
from google.colab import files
import os

print("Silakan pilih berkas 'tokopedia_preprocessed.csv' untuk diunggah:")
uploaded = files.upload()
for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))
    os.rename(fn, 'tokopedia_preprocessed.csv')

## 3. Impor Pustaka dan Siapkan Dataset

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Memuat data
df = pd.read_csv('tokopedia_preprocessed.csv')
df = df.dropna(subset=['review_text', 'label'])

print(f"Total data ulasan: {len(df)}")
dist = df['label'].value_counts()
print(f"Ulasan Asli (0) : {dist.get(0, 0)}")
print(f"Ulasan Palsu (1): {dist.get(1, 0)}")

### Buat Class PyTorch Dataset & Setup Tokenizer
Kita memuat model tokenizer `indobenchmark/indobert-base-p1`.

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Menggunakan 15,000 baris agar pelatihan selesai dengan sangat cepat namun akurasi tetap tinggi.
# Jika ingin melatih seluruh data, silakan hilangkan pembatasan .sample()
df_train_sampled = df.sample(n=min(20000, len(df)), random_state=42).reset_index(drop=True)

X = df_train_sampled['review_text'].astype(str).tolist()
y = df_train_sampled['label'].astype(int).tolist()

# Membagi data
train_texts, val_texts, train_labels, val_labels = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Load Tokenizer
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenizing
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

train_dataset = ReviewDataset(train_encodings, train_labels)
val_dataset = ReviewDataset(val_encodings, val_labels)

print("Dataset PyTorch berhasil siap!")

## 4. Konfigurasi Model & Metrik Evaluasi

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Memuat model pre-trained IndoBERT untuk Sequence Classification
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Cek ketersediaan GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Model dimuat di perangkat: {device}")

## 5. Mulai Pelatihan (Training)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# Mulai fine-tuning!
trainer.train()

## 6. Simpan Model & Kompresi Berkas
Setelah pelatihan selesai, kita akan menyimpan bobot model terbaik dan mengompresinya agar mudah diunduh ke komputer lokal Anda untuk dideploy.

In [ ]:
# Simpan model akhir
model.save_pretrained('./indobert_final')
tokenizer.save_pretrained('./indobert_final')

# Kompres menjadi zip
!zip -r indobert_model.zip ./indobert_final

print("Selesai! Berkas indobert_model.zip siap diunduh dari tab file samping kiri Colab Anda!")